# Creating a bubble plot visualization to demonstrate parameter values

In [1]:
import os

import numpy as np
import pandas as pd
from scipy.stats import t
from pyreadr import read_r
import plotly.graph_objects as go

from shared.wald.from_data.numpy import wald

In [2]:
PARAMETER_ESTIMATES_PATH = 'params/'

affective_motives = np.array(sorted([

    ### Affective motive
    'conversationalist', 'how_enjoyable', 'i_like_you', 'i_felt_close_to_my_partner', 'i_think_your_status', 'i_would_like_to_become_friends',
    # 'overall_affect', 'conv_leader', 'best_affect', 'responsive_1','responsive_2','responsive_3',
    'you_are_good_listener', 'you_are_kind', 'you_are_quickwitted','you_are_warm','you_like_me',

    #test affectives!
    'you_are_intelligent', 'you_are_friendly',

]), dtype=object)
cognitive_motives = np.array(sorted([

    ### Cognitive motives
    # 'anticipated_each_other_sr6', 'developed_joint_perspective_sr2', 'thoughts_became_more_alike_sr5',
    # 'in_common',

    #test cognitives!
    'shared_reality', 'our_thoughts_synced_up_sr1', 'saw_world_in_same_way_sr8',

]), dtype=object)

In [3]:
renamed_affective = [
    '_'.join([w for w in label.split('_') if len(w) > 1])
    for label in affective_motives
]

renamed_cognitive = [
    '_'.join([w for w in label.split('_') if len(w) > 1])
    for label in cognitive_motives
]

In [4]:
# color_choices = ['#6D8196', '#4A4A4A']
# color_choices = ['#6D8196', '#000000']
color_choices = ['#FFA896', '#38000A']
minimum_bubble_size = 2
scale_params_by = 1000
maximum_bubble_size = 40

from_last_k = 500
resample_n = 500
# resample_n = from_last_k
vis_dif = .1

#### Create data structure

##### Affective values

In [5]:
affective_betas, affective_statistics = [], []

In [ ]:
for f in affective_motives:
    data = read_r(os.path.join(PARAMETER_ESTIMATES_PATH, '{}.RData'.format(f)))

    # phi = data['phi'].values.reshape(-1)[-from_last_k:]
    # phi = np.random.choice(phi, size=(resample_n,), replace=False)
    # std_phi = phi.std()
    # phi_tstat = phi.mean() / (std_phi/np.sqrt(phi.shape[0]-1))


    beta = data['beta'].values.reshape(-1)[-from_last_k:]
    beta = np.random.choice(beta, size=(resample_n,), replace=False)
    std_beta = beta.std()
    beta_tstat = beta.mean() / (std_beta/np.sqrt(dof))

    dof = beta.shape[0]-1

    affective_betas += [
        {
            'var': f,
            'dof': dof,

            'phi': data['phi'].mean().item(),
            'std(phi)': std_phi,
            't(phi)': phi_tstat.item(),
            'p(phi)': (1-t.cdf(np.abs(phi_tstat), df=dof)).item()*2,

            'beta': data['beta'].mean().item(),
            'std(beta)': std_beta,
            't(beta)': beta_tstat.item(),
            'p(beta)': (1-t.cdf(np.abs(beta_tstat), df=dof)).item()*2
        }
    ]

affective_betas = pd.DataFrame(affective_betas)

In [ ]:
affective_betas.head(n=20)

##### Cognitive values

In [5]:
cognitive_betas, cognitive_statistics = [], []

In [6]:
for f in cognitive_motives:
    data = read_r(os.path.join(PARAMETER_ESTIMATES_PATH, '{}.RData'.format(f)))

    phi = data['phi'].values.reshape(-1)[-from_last_k:]
    phi = np.random.choice(phi, size=(resample_n,), replace=False)
    dof = phi.shape[0]-1
    std_phi = phi.std()
    phi_tstat = phi.mean() / (std_phi/np.sqrt(phi.shape[0]-1))


    beta = data['beta'].values.reshape(-1)[-from_last_k:]
    beta = np.random.choice(beta, size=(resample_n,), replace=False)
    dof = beta.shape[0]-1
    std_beta = beta.std()
    beta_tstat = beta.mean() / (std_beta/np.sqrt(dof))

    cognitive_betas += [
        {
            'var': f,
            'dof': dof,

            'phi': data['phi'].mean().item(),
            'std(phi)': std_phi,
            't(phi)': phi_tstat.item(),
            'p(phi)': (1-t.cdf(np.abs(phi_tstat), df=dof)).item()*2,

            'beta': data['beta'].mean().item(),
            'std(beta)': std_beta,
            't(beta)': beta_tstat.item(),
            'p(beta)': (1-t.cdf(np.abs(beta_tstat), df=dof)).item()*2
        }
    ]

cognitive_betas = pd.DataFrame(cognitive_betas)

In [7]:
cognitive_betas.head(n=20)

,var,dof,phi,std(phi),t(phi),p(phi),beta,std(beta),t(beta),p(beta)
0,our_thoughts_synced_up_sr1,499,0.130140,0.049015,58.011675,0.0,0.003267,0.588091,0.410338,0.681734
1,saw_world_in_same_way_sr8,499,0.223020,0.047307,104.870980,0.0,-0.010118,0.585088,0.227616,0.820038
2,shared_reality,499,0.166231,0.038558,97.522647,0.0,-0.026649,0.569218,-2.359037,0.018707


#### Merging docs

In [ ]:
affective_betas['motive'] = 'affective'
cognitive_betas['motive'] = 'cognitive'

In [ ]:
dataset = pd.concat([
    # affective_betas,
    cognitive_betas
], ignore_index=True)
dataset.head(n=30)

#### Saving csvs

In [ ]:
dataset.to_csv('/Users/zacharyrosen/Desktop/ICLASP_vars.csv', index=False, encoding='utf-8')

In [ ]:
cognitive_betas.to_csv('cognitive-motives.csv', index=False, encoding='utf-8')
affective_betas.to_csv('affective-motives.csv', index=False, encoding='utf-8')

## Bubble plots

#### Cognitive Motives

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y = np.array([2]*len(cognitive_betas)),
        x = np.array(list(range(len(cognitive_betas)))),
        text = ['ɸ: {}'.format(np.format_float_scientific(v,precision=2)) for v in cognitive_betas['phi']],
        mode='markers',
        name='self',
        legendgroup='self',
        showlegend=False,
        marker=dict(
            size=cognitive_betas['phi'].values.__abs__()*scale_params_by,
            sizemode='area',
            sizeref=2*cognitive_betas['phi'].values.__abs__().max()*scale_params_by/(maximum_bubble_size**2),
            sizemin=minimum_bubble_size,
            color=[color_choices[int(i>0)] for i in cognitive_betas['phi']]
        )
    )
)

fig.add_trace(
    go.Scatter(
        y = np.array([1]*len(cognitive_betas)),
        x = np.array(list(range(len(cognitive_betas)))),
        text = ['β: {}'.format(np.format_float_scientific(v,precision=2)) for v in cognitive_betas['beta']],
        mode='markers',
        name='other',
        legendgroup='other',
        showlegend=False,
        marker=dict(
            size=cognitive_betas['beta'].values.__abs__()*scale_params_by,
            sizemode='area',
            sizeref=2*cognitive_betas['beta'].values.__abs__().max()*scale_params_by/(maximum_bubble_size**2),
            sizemin=minimum_bubble_size,
            color=[color_choices[int(i>0)] for i in cognitive_betas['beta']]
        )
    )
)

fig.update_layout(
    template='plotly_white',
    yaxis=dict(
        tickmode='array',
        tickvals=[1,2],
        ticktext=['other', 'self'],
    ),
    xaxis=dict(
        tickmode='array',
        tickvals=list(range(len(cognitive_betas))),
        ticktext=renamed_cognitive,
    ),
    height=260
)

fig.show()

In [ ]:
fig.write_html('cognitive-motives.html')

#### Affective Motives

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y = np.array([2]*len(affective_betas)),
        x = np.array(list(range(len(affective_betas)))),
        text = ['ɸ: {}'.format(np.format_float_scientific(v,precision=2)) for v in affective_betas['phi']],
        mode='markers',
        name='self',
        legendgroup='self',
        showlegend=False,
        marker=dict(
            size=affective_betas['phi'].values.__abs__()*scale_params_by,
            sizemode='area',
            sizeref=2*affective_betas['phi'].values.__abs__().max()*scale_params_by/(maximum_bubble_size**2),
            sizemin=minimum_bubble_size,
            color=[color_choices[int(i>0)] for i in affective_betas['phi']]
        )
    )
)

fig.add_trace(
    go.Scatter(
        y = np.array([1]*len(affective_betas)),
        x = np.array(list(range(len(affective_betas)))),
        text = ['β: {}'.format(np.format_float_scientific(v,precision=2)) for v in affective_betas['beta']],
        mode='markers',
        name='other',
        legendgroup='other',
        showlegend=False,
        marker=dict(
            size=affective_betas['beta'].values.__abs__()*scale_params_by,
            sizemode='area',
            sizeref=2*affective_betas['beta'].values.__abs__().max()*scale_params_by/(maximum_bubble_size**2),
            sizemin=minimum_bubble_size,
            color=[color_choices[int(i>0)] for i in affective_betas['beta']]
        )
    )
)

fig.update_layout(
    template='plotly_white',
    yaxis=dict(
        tickmode='array',
        tickvals=[1,2],
        ticktext=['other', 'self'],
    ),
    xaxis=dict(
        tickmode='array',
        tickvals=list(range(len(affective_betas))),
        ticktext=renamed_affective,
    ),
    height=300
)

fig.show()

In [ ]:
fig.write_html('affective-motives.html')

## Lolipop plots

In [ ]:
# colors = [
#     'purple',
#     'blue'
# ]

colors = [
    '#005587',
    '#FFB81C'
]

In [ ]:
fig = go.Figure()

# abs_limit = (np.abs(dataset['beta'] / dataset['std(beta)']) < 5)
abs_limit = np.array([True]*len(dataset))
ct = 0
for j,motive_type in enumerate(dataset['motive'].unique()):
    sel = (dataset['motive'] == motive_type) & abs_limit
    sub = dataset.loc[sel]

    showlegend_at = sub.loc[(sub['p(beta)'] < .05)].index.values
    if len(showlegend_at):
        showlegend_at = showlegend_at[0]
    else:
        showlegend_at = sub.index.values[0]

    showlegend = True
    for i in sub.index:
        ct+=1
        showlegend = (i==showlegend_at).item()
        length_of_marker = dataset['beta'].loc[i] #/ dataset['std(beta)'].loc[i]

        ## line
        fig.add_trace(
            go.Scatter(
                x=sorted([0,length_of_marker]),
                y=[ct,ct],
                mode='lines',
                name=motive_type+'_line',
                legendgroup=motive_type,
                showlegend=False,
                line=dict(
                    width=3,
                    color=colors[j]
                ),
            )
        )

        ## marker
        if (dataset['p(beta)'].loc[i] > .05):
            fig.add_trace(
                go.Scatter(
                    x=[length_of_marker],
                    y=[ct],
                    mode='markers',
                    name=motive_type,
                    legendgroup=motive_type,
                    showlegend=showlegend,#showlegend,
                    marker=dict(
                        size=15,
                        line=dict(
                            width=3*(dataset['p(beta)'].loc[i] > .05),
                            color=colors[j]
                        ),
                        color='white'
                    )
                )
            )
        else:
            fig.add_trace(
                go.Scatter(
                    x=[length_of_marker],
                    y=[ct],
                    mode='markers',
                    name=motive_type,
                    legendgroup=motive_type,
                    showlegend=showlegend,#showlegend,
                    marker=dict(
                        size=15,
                        line=dict(
                            width=3,
                            color=colors[j]
                        ),
                        color=colors[j]
                    )
                )
            )

fig.add_vline(x=0, layer='below', line_width=3)
fig.update_yaxes(
    tickmode='array',
    tickvals=list(range(1,abs_limit.sum()+1)),
    ticktext=dataset['var'].loc[abs_limit].values
)
# fig.update_xaxes(range=[(-dataset['beta'].__abs__().max())-vis_dif,dataset['beta'].__abs__().max()+vis_dif])
fig.update_layout(
    template='ygridoff',
    # width=600,
    margin=dict(l=225,r=10),
    xaxis=dict(automargin=True)
)
fig.show()

In [ ]:
# fig.write_html('lolipop-other.html')
# fig.write_html('lolipop-other-all.html')
fig.update_layout(margin=dict(l=225,r=10,t=50,b=50))
fig.write_html('/Users/zacharyrosen/Desktop/lolipop-other-all-3.html')

In [ ]:
# fig.write_html('lolipop-other-zoomed.html')
# # fig.write_html('/Users/zacharyrosen/Desktop/lolipop-other-2.html')

### Hypothesis examples

In [ ]:
colors = [
    '#1d3c45',
    '#d2601a'
]

##### Lollipop

In [ ]:
fig = go.Figure()

In [ ]:
# faster
fig.add_trace(
    go.Scatter(
        x=[-1,0],
        y=[1,1],
        mode='lines',
        name='fast changes'+'_line',
        legendgroup='fast changes',
        showlegend=False,
        line=dict(
            width=3,
            color=colors[0]
        ),
    )
)
fig.add_trace(
    go.Scatter(
        x=[-1],
        y=[1],
        mode='markers',
        name='fast changes',
        legendgroup='fast changes',
        showlegend=True,#showlegend,
        marker=dict(
            size=15,
            line=dict(
                width=3*(dataset['p(beta)'].loc[i] > .05),
                color=colors[0]
            ),
            color='white'
        )
    )
)

# slower
fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[-1,-1],
        mode='lines',
        name='slow changes'+'_line',
        legendgroup='slow changes',
        showlegend=False,
        line=dict(
            width=3,
            color=colors[1]
        ),
    )
)
fig.add_trace(
    go.Scatter(
        x=[1],
        y=[-1],
        mode='markers',
        name='slow changes',
        legendgroup='slow changes',
        showlegend=True,#showlegend,
        marker=dict(
            size=15,
            line=dict(
                width=3,
                color=colors[1]
            ),
            color='white'
        )
    )
)

In [ ]:
fig.update_layout(
    template='ygridoff',
    legend=dict(
        x=0,
        y=1,
        xanchor='left',  # Anchors the left side of the legend box to the x position
        yanchor='top'    # Anchors the top side of the legend box to the y position
    ),
    # Optional: adjust margins if the legend overlaps the plot title or top area
    margin=dict(t=100),
    yaxis=dict(showticklabels=False)
)
fig.show()

In [ ]:
fig.write_html('example_hypotheses/hypthosis-lollipop.html')

##### exponential growth

In [ ]:
time = np.linspace(1,50,100)

Fast rate of change

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=time,
        y=time**(-2),
        mode='lines',
        line=dict(
            color=colors[0],
            width=10
        )
    )
)

In [ ]:
fig.update_layout(
    template='ygridoff',
    yaxis=dict(showticklabels=False)
)
fig.show()

In [ ]:
fig.write_html('example_hypotheses/curve-fast.html')

Slow rate of change

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=time,
        y=time**(15),
        mode='lines',
        line=dict(
            color=colors[1],
            width=10
        )
    )
)

In [ ]:
fig.update_layout(
    template='ygridoff',
    yaxis=dict(showticklabels=False)
)
fig.show()

In [ ]:
fig.write_html('example_hypotheses/curve-slow.html')

## Fast Divergence versus Durative Convergence

### Load CANDOR metadata (?)

In [ ]:
CANDOR_META_DATA_PATH = '../../ComplexityScienceOfEverydayConvos/english/data/all-candor-meta_data.csv'
os.path.exists(CANDOR_META_DATA_PATH)

In [ ]:
df = pd.read_csv(CANDOR_META_DATA_PATH)[['user_id', 'convo_id']+affective_motives.tolist()+cognitive_motives.tolist()]
df.head()

### T-test and cluster

In [ ]:
from scipy.stats import t

In [ ]:
def tstat(x1,x2,s1,s2,k1,k2):
    varo, varn = s1**2, s2**2
    svaro, svarn = varo/k1, varn/k2
    svaro_svarn = svaro + svarn

    tstat = (x1-x2) / np.sqrt(svaro_svarn)

    dof_denom = ((svaro**2) / (k1-1)) + ((svarn**2) / (k2-1))
    dof = (svaro_svarn**2) / dof_denom

    return tstat, dof

def Pt(x1,x2,s1,s2,k1,k2):
    ts, dof = tstat(x1,x2,s1,s2,k1,k2)
    return t.sf(ts.__abs__(), dof)

In [ ]:
minx, mins = dataset[['beta', 'std(beta)']].loc[
    (dataset['beta'] == dataset['beta'].loc[dataset['p(beta)'] < .01].min())
].values[0]
minx, mins

In [ ]:
maxx, maxs = dataset[['beta', 'std(beta)']].loc[
    dataset['beta']==dataset['beta'].loc[dataset['p(beta)'] < .01].max()
].values[0]
maxx, maxs

In [ ]:
dataset['min_pole_p'] = [Pt(x,minx,sd,mins,50,50) for x,sd in dataset[['beta', 'std(beta)']].values]
dataset['max_pole_p'] = [Pt(x,maxx,sd,maxs,50,50) for x,sd in dataset[['beta', 'std(beta)']].values]
dataset['normalizer'] = dataset[['min_pole_p', 'max_pole_p']].values.sum(axis=-1)


dataset[['min_pole_p', 'max_pole_p']] = dataset[['min_pole_p', 'max_pole_p']].values / dataset['normalizer'].values.reshape(-1,1)

In [ ]:
dataset[['var', 'beta', 'min_pole_p', 'max_pole_p', 'normalizer']].head(n=20)

In [ ]:
dataset.to_csv('/Users/zacharyrosen/Desktop/ICLASP-results.csv', index=False, encoding='utf-8')